# Module 9: File Handling

**Utrains Python Fundamentals** &middot; lab notebook

*Read and write files, work safely with with, and handle JSON, including real API responses.*

## What you will be able to do by the end

- Pick the right file mode for what you are about to do
- Read a whole file, one line, or every line as a list
- Prefer with open(...) so files always get closed
- Turn Python data into JSON and back, as a string and as a file
- Locate a config file on disk with pathlib

## How this notebook is organised

It follows the Module 9 slide deck, slide by slide.

Run every cell in order with **Shift + Enter**. Read the note above each block,
then run the code and compare what you see with what you expected.

Two cells are marked **Your turn**. They contain `____` where a piece of the
syntax is missing, so they will fail if you run them as they are. That is
deliberate. Replace each `____`, then run the cell until it succeeds.

The last section is the **Lab**: a short task with no code written for you.

## What this notebook assumes

Modules 1 to 8. `import` appears throughout because `os`, `json` and `pathlib` need it; the deck flags this too, and Module 10 explains `import` properly.

### Before you start: the scratch folder

Everything in this notebook writes into a `scratch/` folder next to it, so
nothing else on your machine is touched. `scratch/` is in `.gitignore`, so none
of it will end up in a commit. Run this cell first.

In [ ]:
from pathlib import Path

WORK = Path("scratch")
WORK.mkdir(exist_ok=True)

print("writing files into:", WORK.resolve())

## Slide 2 &middot; What Is File Handling?

File handling lets your program read and write data outside of memory, so it
survives after the script finishes running.

**Two verbs, one function.** `open()` handles both reading and writing. What it
does depends entirely on the mode you pass it.

In [ ]:
# write something to disk
file = open(WORK / "notes.txt", "w")
file.write("Meeting at 3pm")
file.close()

# read it back later, even after the program restarts
file = open(WORK / "notes.txt", "r")
print(file.read())
file.close()

## Slide 3 &middot; File Modes

You choose a mode when you open a file, which controls what you are allowed to
do with it.

| Mode | Meaning |
|---|---|
| `"r"` | Read. The default mode. |
| `"w"` | Write. **Overwrites** anything already there. |
| `"a"` | Append. Adds new content to the end. |
| `"x"` | Create. Fails if the file already exists. |
| `"rb"` / `"wb"` | Same as r and w, for binary data. |

## Slide 4 &middot; Opening, Writing, and Reading

Write to a file, then open it again separately to read it back.

**Do not forget `close()`.** Skipping it can leave a file locked or lose
unsaved data. The `with` statement, coming up shortly, removes this risk
entirely.

In [ ]:
file = open(WORK / "sample.txt", "w")
file.write("Hello, this is a test file.")
file.close()

file = open(WORK / "sample.txt", "r")
content = file.read()
print(content)
file.close()

## Slide 5 &middot; Reading Line by Line

You do not have to read a whole file at once. Pull one line, or every line as a
list.

In [ ]:
file = open(WORK / "sample.txt", "r")
line = file.readline()          # reads a single line
file.close()
print("readline :", repr(line))

file = open(WORK / "sample.txt", "r")
lines = file.readlines()        # reads all lines into a list
file.close()
print("readlines:", lines)

## Slide 6 &middot; The with Statement

Opening a file with `with` automatically closes it for you, even if an error
happens partway through. This is the pattern to reach for by default.

**No `close()` needed.** `with` removes the risk of a locked file or lost data
entirely, so prefer it over `open()` / `close()` pairs.

In [ ]:
with open(WORK / "sample.txt", "r") as file:
    content = file.read()
    print(content)

# the file is already closed here, no need to call close()
print("closed?", file.closed)

## Slide 7 &middot; Writing Multiple Lines and Appending

`writelines()` takes a list of strings and writes each one. Appending with
`"a"` adds to the end without touching what is already there.

**Remember the newline.** `writelines()` does not add `\n` for you. Each
string in the list needs its own.

In [ ]:
lines = ["Line 1\n", "Line 2\n", "Line 3\n"]

with open(WORK / "output.txt", "w") as file:
    file.writelines(lines)

with open(WORK / "output.txt", "a") as file:
    file.write("This line will be appended.\n")

with open(WORK / "output.txt", "r") as file:
    print(file.read())

---

### Your turn 1

Append an incident summary to a running log, then read the whole log back. Pick the mode that adds to the end rather than wiping the file.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
log_path = WORK / "incidents.log"

# TODO: choose the mode that ADDS to the end rather than overwriting.
with open(log_path, ____) as f:
    f.write("INC-4412 database connection failures resolved\n")

# TODO: choose the mode that reads.
with open(log_path, ____) as f:
    print(f.read())

Run that cell two or three times. The log grows each time, which is exactly what append mode is for. Change the mode to `"w"`, run it again, and watch the earlier lines disappear.

## Slide 8 &middot; Working with Binary Files

Images and other non-text files need `"rb"` and `"wb"`, the binary versions of
read and write mode. Binary content comes back as bytes rather than text.

In [ ]:
# Make a small binary file so there is something real to copy.
with open(WORK / "image.jpg", "wb") as f:
    f.write(bytes(range(40)))

with open(WORK / "image.jpg", "rb") as file:
    data = file.read()
    print("Binary content:", data[:20])

with open(WORK / "copy.jpg", "wb") as new_file:
    new_file.write(data)

print("copy written, same size:", (WORK / "copy.jpg").stat().st_size == len(data))

## Slide 9 &middot; Checking and Deleting Files

Checking whether a file exists and deleting one both live in a module called
`os`, which ships with Python.

**First time seeing `import`?** Module 10 explains it in full. The short
version: it brings in code someone else already wrote, used with a dot, like
`os.path.exists()`.

In [ ]:
import os

target = WORK / "sample.txt"

if os.path.exists(target):
    print("File exists")
else:
    print("File not found")

os.remove(target)
print("after remove, exists?", os.path.exists(target))

## Slide 10 &middot; Working with JSON

JSON is a text format almost every API uses to send and receive data. It maps
directly onto a Python dictionary.

**Two functions to know.** `json.dumps` turns a Python object into a JSON
string. `json.loads` turns a JSON string back into a Python object.

In [ ]:
import json

person = {"name": "Alice", "age": 25}

as_text = json.dumps(person)
print(as_text)
print(type(as_text))

back_to_dict = json.loads(as_text)
print(back_to_dict["name"])
print(type(back_to_dict))

## Slide 11 &middot; Parsing a Real JSON Response

A model API response arrives as JSON text over the network. Parsing it is the
same skill applied to something you did not write yourself.

**Follow the brackets.** `data["content"][0]["text"]` is a dictionary, then a
list, then a dictionary again, one step at a time. That is exactly the nested
structure from Module 6.

In [ ]:
raw_response = '''
{
  "model": "claude-sonnet-4-6",
  "content": [{"type": "text", "text": "The capital of France is Paris."}],
  "usage": {"input_tokens": 12, "output_tokens": 8}
}
'''

data = json.loads(raw_response)

print(data["content"][0]["text"])
print(data["usage"]["output_tokens"])

## Slide 12 &middot; Reading and Writing JSON Files

`json.load` and `json.dump` work the same way, but read from and write directly
to a file, so you never build the string by hand.

**dump vs dumps.** `dump` writes to a file object. `dumps`, with an `s`,
returns a string. The same pattern applies to `load` and `loads`.

In [ ]:
conversation = [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "Paris."},
]

with open(WORK / "conversation.json", "w") as f:
    json.dump(conversation, f, indent=2)

with open(WORK / "conversation.json", "r") as f:
    loaded = json.load(f)

print(loaded[0]["content"])
print("turns saved:", len(loaded))

---

### Your turn 2

Save a dictionary of resource tags to a JSON file and read one value back. Watch carefully which of the four json functions each step needs.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
tags = {"env": "prod", "owner": "platform-team", "cost-center": "CC-1180"}

# TODO: write to a FILE, then read from a FILE. Mind the s.
with open(WORK / "tags.json", "w") as f:
    json.____(tags, f, indent=2)

with open(WORK / "tags.json", "r") as f:
    restored = json.____(f)

print(restored["owner"])

## Slide 13 &middot; Finding a Config File with pathlib

`pathlib` is another built in module for working with file paths. This pattern
searches a few likely folders for a configuration file.

**AI framing.** This is exactly how AI projects locate API keys stored outside
the code, in a `.env` file.

In [ ]:
from pathlib import Path

here = Path.cwd().resolve()
loaded_from = None

for candidate in [
    here / ".env",
    here.parent / ".env",
    here.parent.parent / ".env",
]:
    if candidate.is_file():
        loaded_from = candidate
        break

print("searched from:", here)
print("Would load .env from:", loaded_from)

### Reading a file that ships with this repo

`data/servers.txt` sits one folder up from this notebook, with one server name
per line. Reading a real file is the same code, just a different path.

In [ ]:
servers_file = Path("..") / "data" / "servers.txt"

with open(servers_file) as f:
    for line in f:
        print("-", line.strip())

---

## Lab: A server inventory round trip


Read `../data/servers.txt`, which holds one server name per line.

Turn it into a list of dictionaries, where each entry has a `name`, a `region`
taken from the part of the name after the first hyphen that follows the number,
and a `status` of `"unknown"`.

Save that list to `scratch/inventory.json` with an indent of 2, then read it
back from disk into a fresh variable, and print how many servers you recovered
and the name of the last one.

Finish by appending one audit line to `scratch/audit.log` recording how many
servers were processed. Run the whole lab twice: the audit log should have two
lines while the inventory JSON still holds the right count.


**Done when:**

- [ ] The text file is read with a with statement, not open/close
- [ ] Each line becomes a dictionary in a list
- [ ] json.dump writes the file and json.load reads it back
- [ ] The audit log uses append mode and grows on a second run

Write your answer in the cell below. There is no starter code on purpose.

In [ ]:
# Your lab answer goes here.

---

## Practice exercises

These are the four exercises from the module's practice slide, word for word.

1. Write a script that reads a list of server names from a text file, one per line, and prints each one.
2. Save a dictionary of cloud resource tags to a JSON file, then read it back and print one of the tag values.
3. Append a new incident summary line to a running incident log file each time the script runs.
4. Save a short conversation history (a list of role/content dictionaries) to a .json file, then reload it and print the last message.

---

## Module complete

You can now read, write and manage files, and work with JSON data confidently.

*Utrains &middot; support@utrains.org &middot; https://utrains.org*